<a href="https://colab.research.google.com/github/ssk-algoverse/sae-binding/blob/main/aunt_name_binding.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from torch import nn
import torch
from torch.utils.data import DataLoader, TensorDataset, random_split
from tabulate import tabulate
import pandas as pd
import torch.nn.functional as F
from tabulate import tabulate
from IPython.display import display
from torch.nn.functional import cosine_similarity
import random
from tqdm import tqdm


DEVICE = "cuda"
LAYER = 12


class ScalarAdd(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.x = torch.nn.Parameter(torch.zeros((), dtype=torch.bfloat16))

    def forward(self, n, t):
        return n - self.x * t


class LinearAdd(torch.nn.Module):
    def __init__(self, d=2304):
        super().__init__()
        self.X = torch.nn.Linear(d, d, bias=False).to(torch.bfloat16)

    def forward(self, n, t):
        return n - self.X(t)


import torch
from typing import Tuple, Optional


def train_model(
    model: torch.nn.Module,
    train_loader: torch.utils.data.DataLoader,
    val_loader: torch.utils.data.DataLoader,
    *,
    n_epochs: int = 200,
    lr: float = 1e-2,
    patience: int = 20,  # epochs to wait for improvement
    min_delta: float = 0.0,  # minimum improvement regarded as progress
    device: Optional[torch.device] = None,
) -> Tuple[float, float, torch.nn.Module]:
    """
    Train `model` with early stopping.

    Parameters
    ----------
    model : nn.Module
    train_loader, val_loader : DataLoader
        The second loader is used for validation *and* is what the function
        ultimately returns as the “test” loss (keep them separate if you prefer).
    n_epochs : int
        Maximum number of epochs.
    lr : float
        Learning-rate for Adam.
    patience : int
        Stop if `val_loss` hasn’t improved for this many epochs.
    min_delta : float
        Required relative improvement (`old - new > min_delta`) to reset patience.
    device : torch.device or None
        If given, the model and batches are moved to that device.

    Returns
    -------
    train_mse : float
    val_mse   : float
    model     : nn.Module  (best model parameters are restored)
    """
    if device is not None:
        model.to(device)

    opt = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = torch.nn.MSELoss()

    best_loss = float("inf")
    best_state = None
    wait = 0

    def mse(loader):
        """Average MSE over *all* batches in `loader`."""
        model.eval()
        total_loss, n_obs = 0.0, 0
        with torch.no_grad():
            for n_b, t_b, y_b in loader:
                if device is not None:
                    n_b, t_b, y_b = n_b.to(device), t_b.to(device), y_b.to(device)
                pred = model(n_b, t_b)
                batch_loss = loss_fn(pred, y_b).item()
                total_loss += batch_loss * len(y_b)
                n_obs += len(y_b)
        model.train()
        return total_loss / n_obs

    for epoch in range(1, n_epochs + 1):
        for n_b, t_b, y_b in train_loader:
            if device is not None:
                n_b, t_b, y_b = n_b.to(device), t_b.to(device), y_b.to(device)
            opt.zero_grad(set_to_none=True)
            loss = loss_fn(model(n_b, t_b), y_b)
            loss.backward()
            opt.step()

        # ---- end of epoch: check validation loss ----
        val_loss = mse(val_loader)

        if val_loss + min_delta < best_loss:  # improvement
            best_loss = val_loss
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
            wait = 0
        else:
            wait += 1
            if wait >= patience:
                break

    # Restore best weights before returning
    if best_state is not None:
        model.load_state_dict(best_state)

    return mse(train_loader), best_loss, model


@torch.no_grad()
def mean_cosine(loader, model):
    model.eval()
    preds, targs = [], []
    for x_name, x_prefix, y in loader:
        x_name, x_prefix, y = x_name.to(DEVICE), x_prefix.to(DEVICE), y.to(DEVICE)
        preds.append(model(x_name, x_prefix))
        targs.append(y)
    preds = torch.cat(preds, 0)
    targs = torch.cat(targs, 0)
    return F.cosine_similarity(preds, targs, dim=-1).mean().item()


def fit_additive_models(
    name_hidden_states,
    prefix_hidden_states,
    prefix_name_hidden_states,
    n_epochs: int = 200,
    lr: float = 1e-2,
    train_ratio: float = 0.8,
    seed: int = 0,
):

    n, t, c = name_hidden_states, prefix_hidden_states, prefix_name_hidden_states

    # ------------------------------------------------------------
    # Build every (name, prefix) pairing
    # ------------------------------------------------------------
    name_idx = torch.arange(n.size(0)).repeat(t.size(0))
    prefix_idx = torch.arange(t.size(0)).repeat_interleave(n.size(0))
    X_name = n[name_idx]
    X_prefix = t[prefix_idx]
    y = c  # ground-truth joint states

    dataset = TensorDataset(X_name, X_prefix, y)

    # ------------------------------------------------------------
    # Train / test split
    # ------------------------------------------------------------
    name_hidden_states, prefix_hidden_states[0:1], prefix_name_hidden_states[
        : len(names)
    ]
    train_len = int(len(dataset) * train_ratio)
    test_len = len(dataset) - train_len
    g = torch.Generator().manual_seed(seed)
    train_ds, test_ds = random_split(dataset, [train_len, test_len], generator=g)
    train_loader = DataLoader(train_ds, batch_size=len(train_ds))
    test_loader = DataLoader(test_ds, batch_size=len(test_ds))

    # ------------------------------------------------------------
    # Fit scalar-additive model
    # ------------------------------------------------------------
    scalar_train_mse, scalar_test_mse, scalar_model = train_model(
        ScalarAdd().to(DEVICE), train_loader, test_loader, n_epochs=n_epochs, lr=lr
    )
    scalar_train_cos = mean_cosine(train_loader, scalar_model)
    scalar_test_cos = mean_cosine(test_loader, scalar_model)

    # ------------------------------------------------------------
    # Fit linear-additive model
    # ------------------------------------------------------------
    linear_train_mse, linear_test_mse, linear_model = train_model(
        LinearAdd().to(DEVICE), train_loader, test_loader, n_epochs=n_epochs, lr=lr
    )
    linear_train_cos = mean_cosine(train_loader, linear_model)
    linear_test_cos = mean_cosine(test_loader, linear_model)

    # ------------------------------------------------------------
    # Return metrics + trained model objects
    # ------------------------------------------------------------
    return {
        "scalar": {
            "train_mse": scalar_train_mse,
            "test_mse": scalar_test_mse,
            "train_cos": scalar_train_cos,
            "test_cos": scalar_test_cos,
            "model": scalar_model,
        },
        "linear": {
            "train_mse": linear_train_mse,
            "test_mse": linear_test_mse,
            "train_cos": linear_train_cos,
            "test_cos": linear_test_cos,
            "model": linear_model,
        },
    }


if __name__ == "__main__":
    import math
    import statistics as stats

    # ------------------------------------------------------------------
    # Helper for 95% CI (df = n-1, here n = 10 → t_{0.975,9} ≈ 2.262)
    # ------------------------------------------------------------------
    def mean_ci(values, t_crit: float = 2.262):
        """Return (mean, half-width 95 % CI) for a list/1-D Tensor."""
        m = (
            float(torch.tensor(values).mean().item())
            if torch.is_tensor(values[0])
            else float(stats.mean(values))
        )
        if len(values) == 1:
            return m, 0.0  # degenerate CI when only a single sample
        sd = (
            float(torch.tensor(values).std(unbiased=True).item())
            if torch.is_tensor(values[0])
            else float(stats.stdev(values))
        )
        se = sd / math.sqrt(len(values))
        return m, t_crit * se

    # ------------------------------------------------------------------
    # Load model / tokenizer & pre-compute hidden states (expensive)
    # ------------------------------------------------------------------
    model = (
        AutoModelForCausalLM.from_pretrained("google/gemma-2-2b")
        .to(DEVICE)
        .to(torch.bfloat16)
    )
    tokenizer = AutoTokenizer.from_pretrained("google/gemma-2-2b")

    # Build token tensors exactly as before ---------------------------------
    names = [
        "Anna",
        "Mary",
        "Susan",
        "Linda",
        "Emily",
        "Grace",
        "Helen",
        "Sarah",
        "Julia",
        "Laura",
        "Alice",
        "Betty",
        "Diana",
        "Irene",
        "Karen",
        "Martha",
        "Nancy",
        "Olga",
        "Paula",
        "Ruth",
        "John",
        "Paul",
        "Mark",
        "Luke",
        "James",
        "Peter",
        "David",
        "Robert",
        "Michael",
        "William",
        "Richard",
        "Charles",
        "Thomas",
        "Daniel",
        "Matthew",
        "Anthony",
        "Steven",
        "Andrew",
        "Joshua",
        "Kevin",
        "Brian",
        "Eric",
        "George",
        "Henry",
        "Jack",
        "Louis",
        "Martin",
        "Oscar",
        "Philip",
        "Simon",
        "Sam",
        "Alex",
        "Jane",
        "Zoe",
        "Max",
        "Leo",
        "Eve",
        "Kim",
        "Roy",
        "Ben",
        "Amy",
        "Ella",
        "Liam",
        "Noah",
        "Mia",
        "Chloe",
        "Sophie",
        "Jake",
        "Owen",
        "Sean",
        "Tina",
        "Nina",
        "Joel",
        "Ivan",
        "Alan",
        "June",
        "Dean",
        "Tara",
        "Vera",
        "Gina",
        "Lily",
        "Hope",
        "Rose",
        "Luke",
        "Finn",
        "Jill",
        "Kate",
        "Megan",
        "Scott",
        "Toby",
        "Rosa",
        "Clara",
        "Miles",
        "Derek",
        "Seth",
        "Dana",
        "Rita",
        "Vince",
        "Tess",
        "Joel",
        "Mona",
        "Gail",
        "Wade",
        "Rex",
        "Troy",
        "Lars",
        "Cody",
        "Bruce",
        "Eli",
        "Milo",
        "Nora",
        "Sage",
        "Jude",
        "Omar",
        "Ivy",
        "Reed",
        "Skye",
        "Drew",
        "Hope",
        "Blake",
        "Elle",
        "Tess",
        "Dean",
        "Quinn",
        "Rene",
        "Saul",
        "Eva",
        "Lara",
        "Maya",
        "Nina",
        "Lila",
        "Mila",
        "Sara",
        "Lana",
        "Vera",
        "Rosa",
        "Luca",
        "Ivan",
        "Omar",
        "Hugo",
        "Enzo",
        "Noel",
        "Joel",
        "Alan",
        "Leon",
        "Alex",
        "Anna",
        "Lina",
        "Mira",
        "Elsa",
        "Rene",
        "Tina",
        "Liam",
        "Noah",
        "Milo",
        "Eli",
        "Nora",
        "Ivy",
        "Jude",
        "Reed",
        "Skye",
        "Drew",
        "Hope",
        "Blake",
        "Elle",
        "Tess",
        "Dean",
        "Quinn",
        "Saul",
        "Rex",
        "Troy",
        "Lars",
        "Cody",
        "Bruce",
        "Kim",
        "Roy",
        "Ben",
        "Amy",
        "Ella",
        "Mia",
        "Chloe",
        "Sophie",
        "Jake",
        "Owen",
        "Sean",
        "June",
        "Tara",
        "Gina",
        "Lily",
        "Rose",
        "Finn",
        "Jill",
        "Kate",
        "Megan",
        "Scott",
        "Toby",
        "Clara",
        "Miles",
        "Derek",
        "Seth",
        "Dana",
        "Rita",
        "Vince",
        "Mona",
        "Gail",
        "Wade",
        "Max",
        "Leo",
        "Sam",
        "Jane",
        "Zoe",
        "Oscar",
        "Philip",
        "Simon",
        "Jack",
        "Louis",
        "Martin",
        "Oscar",
        "Philip",
        "Simon",
        "Sam",
        "Alex",
        "Jane",
        "Zoe",
        "Max",
        "Leo",
        "Eve",
        "Kim",
        "Roy",
        "Ben",
        "Amy",
        "Ella",
        "Liam",
        "Noah",
        "Mia",
        "Chloe",
        "Sophie",
        "Jake",
        "Owen",
        "Sean",
        "Tina",
        "Nina",
        "Joel",
        "Ivan",
        "Alan",
        "June",
        "Dean",
        "Tara",
        "Vera",
        "Gina",
        "Lily",
        "Hope",
        "Rose",
        "Luke",
        "Finn",
        "Jill",
        "Kate",
        "Megan",
        "Scott",
        "Toby",
        "Rosa",
        "Clara",
        "Miles",
        "Derek",
        "Seth",
        "Dana",
        "Rita",
        "Vince",
        "Tess",
        "Joel",
        "Mona",
        "Gail",
        "Wade",
        "Rex",
        "Troy",
        "Lars",
        "Cody",
        "Bruce",
        "Eli",
        "Milo",
        "Nora",
        "Sage",
        "Jude",
        "Omar",
        "Ivy",
        "Reed",
        "Skye",
        "Drew",
        "Hope",
        "Blake",
        "Elle",
        "Tess",
        "Dean",
        "Quinn",
        "Rene",
        "Saul",
    ]
    names = list(set(names))
    single_token_names = []
    for name in names:
        toks = tokenizer(name, return_tensors="pt")["input_ids"]
        if toks.size(1) == 2:  # [BOS, token]
            single_token_names.append(name)
    names = single_token_names

    titles = [
        "Mrs",
        "Miss",
        "Ms",
        "Mr",
        "Master",
        "Dr",
        "Prof",
        "Sir",
        "Lady",
        "Lord",
        "Mx",
        "Rev",
        "Fr",
        "Br",
        "Sr",
        "Madam",
        "Dame",
        "Capt",
        "Major",
        "Col",
        "Gen",
        "Hon",
        "Judge",
        "Pres",
        "Gov",
        "Amb",
        "Sec",
        "Pope",
        "Rabbi",
        "Imam",
        "Sheikh",
        "Guru",
        "Sifu",
        "Sensei",
        "Coach",
        "Admiral",
        "Chief",
        "Commander",
        "Warden",
        "Marshal",
        "Constable",
        "Deputy",
        "Agent",
        "Inspector",
        "Detective",
        "Officer",
        "Sergeant",
        "Corporal",
        "Private",
        "Lieutenant",
        "Captain",
        "Major",
        "Colonel",
        "General",
        "Baron",
        "Baroness",
        "Count",
        "Countess",
        "Duke",
        "Duchess",
        "Emperor",
        "Empress",
        "King",
        "Queen",
        "Prince",
        "Princess",
        "Tsar",
        "Czar",
        "Shah",
        "Sultan",
        "Ayatollah",
        "Cardinal",
        "Bishop",
        "Archbishop",
        "Pastor",
        "Minister",
        "Chaplain",
        "Canon",
        "Monsignor",
        "Patriarch",
        "Matriarch",
        "Elder",
        "Brother",
        "Sister",
        "Father",
        "Mother",
        "Abbess",
        "Abbot",
        "Provost",
        "Principal",
        "President",
        "Chancellor",
        "Premier",
        "Senator",
        "Representative",
        "Councillor",
        "Mayor",
        "Sheriff",
        "Marshal",
        "Rector",
        "Prefect",
        "Commodore",
        "Navigator",
        "Pilot",
        "Sailor",
        "Ensign",
        "Cadet",
        "Medic",
        "Nurse",
        "Judge",
        "Justice",
        "Vicar",
        "Deacon",
        "Chaplain",
        "Saint",
        "Prophet",
        "Seer",
        "Sage",
        "Magus",
        "Wizard",
        "Sorcerer",
        "Witch",
        "Warlock",
        "Druid",
        "Shaman",
        "Healer",
        "Alchemist",
        "Guru",
        "Yogi",
        "Monk",
        "Nun",
        "Coach",
        "Agent",
    ]
    titles = list(set(titles))
    single_token_titles = []
    for title in titles:
        toks = tokenizer(title, return_tensors="pt")["input_ids"]
        if toks.size(1) == 2:  # [BOS, token]
            single_token_titles.append(title)
    titles = single_token_titles

    # Random "other" prefixes ---------------------------------------------
    all_token_ids = list(range(4, tokenizer.vocab_size))
    random_token_ids = random.sample(all_token_ids, len(titles))
    other = [tokenizer.decode([tid]).strip() for tid in random_token_ids]

    prefixes = titles + other

    # ------------------------------------------------------------------
    # Prepare tokens (single-token names/prefixes enforced)
    # ------------------------------------------------------------------
    name_tokens = tokenizer(names, return_tensors="pt")["input_ids"]
    assert name_tokens.size(1) == 2, "Use single name tokens only"

    prefix_tokens = tokenizer(titles, return_tensors="pt")["input_ids"]
    prefix_tokens = torch.cat([prefix_tokens] * 2, dim=0)
    prefix_tokens[len(titles) :, 1] = torch.tensor(random_token_ids)
    assert prefix_tokens.size(1) == 2, "Use single prefix tokens only"

    # Convenience function --------------------------------------------------
    def get_hidden_states(model, input_ids, batch_size: int = 32):
        hidden_states = []
        with torch.no_grad():
            for i in tqdm(range(0, input_ids.size(0), batch_size), desc="Batches"):
                batch = input_ids[i : i + batch_size]
                outputs = model(input_ids=batch.to(DEVICE), output_hidden_states=True)
                hidden_states.append(outputs.hidden_states[LAYER].squeeze(0)[:, -1])
        return torch.cat(hidden_states, dim=0)

    # Heavy forward passes: done ONCE --------------------------------------
    name_hidden_states = get_hidden_states(model, name_tokens)
    prefix_hidden_states = get_hidden_states(model, prefix_tokens)

    prefix_name_tokens = torch.cat(
        [
            prefix_tokens.repeat_interleave(len(names), dim=0),
            name_tokens.repeat(len(prefixes), 1),
        ],
        dim=1,
    )
    prefix_name_hidden_states = get_hidden_states(model, prefix_name_tokens)

    print(
        f"Number of names: {len(names)}, Number of titles: {len(titles)}, Number of prefix names: {prefix_name_hidden_states.size(0)}"
    )

    # ------------------------------------------------------------------
    # Repeated training/evaluation loop (n_runs) ------------------------
    # ------------------------------------------------------------------
    n_runs = 10
    conditions = [
        "Title Only",
        "Other Only",
        "Title Only (Random)",
        "Other Only (Random)",
    ]
    metrics = [
        "mse_scalar",
        "mse_linear",
        "cos_scalar",
        "cos_linear",
    ]
    results = {c: {m: [] for m in metrics} for c in conditions}

    for run in tqdm(range(n_runs)):
        # ---- True prefix hidden-state baselines ------------------------
        title_only = fit_additive_models(
            name_hidden_states,
            prefix_hidden_states[: len(titles)],
            prefix_name_hidden_states[: len(titles) * len(names)],
        )
        other_only = fit_additive_models(
            name_hidden_states,
            prefix_hidden_states[len(titles) :],
            prefix_name_hidden_states[len(titles) * len(names) :],
        )

        # ---- Random prefix representations (new each run) -------------
        rand_title_only = fit_additive_models(
            name_hidden_states,
            torch.rand_like(prefix_hidden_states[: len(titles)]),
            prefix_name_hidden_states[: len(titles) * len(names)],
        )
        rand_other_only = fit_additive_models(
            name_hidden_states,
            torch.rand_like(prefix_hidden_states[len(titles) :]),
            prefix_name_hidden_states[len(titles) * len(names) :],
        )

        # ---- Store metrics -------------------------------------------
        mapping = [
            ("Title Only", title_only),
            ("Other Only", other_only),
            ("Title Only (Random)", rand_title_only),
            ("Other Only (Random)", rand_other_only),
        ]
        for cond, res in mapping:
            results[cond]["mse_scalar"].append(res["scalar"]["test_mse"])
            results[cond]["mse_linear"].append(res["linear"]["test_mse"])
            results[cond]["cos_scalar"].append(res["scalar"]["test_cos"])
            results[cond]["cos_linear"].append(res["linear"]["test_cos"])

    # ------------------------------------------------------------------
    # Build final DataFrame with mean ± 95 % CI -------------------------
    # ------------------------------------------------------------------
    table_rows = []
    for cond in conditions:
        m_s, ci_s = mean_ci(results[cond]["mse_scalar"])
        m_l, ci_l = mean_ci(results[cond]["mse_linear"])
        c_s, ci_cs = mean_ci(results[cond]["cos_scalar"])
        c_l, ci_cl = mean_ci(results[cond]["cos_linear"])

        table_rows.append(
            [
                cond,
                f"{m_s:.4f} ± {ci_s:.4f}",
                f"{m_l:.4f} ± {ci_l:.4f}",
                f"{c_s:.4f} ± {ci_cs:.4f}",
                f"{c_l:.4f} ± {ci_cl:.4f}",
            ]
        )

    df = pd.DataFrame(
        table_rows,
        columns=pd.MultiIndex.from_tuples(
            [
                ("", "Title"),
                ("MSE", "Scalar"),
                ("MSE", "Linear"),
                ("Cosine Sim", "Scalar"),
                ("Cosine Sim", "Linear"),
            ]
        ),
    )

    # Pretty print -------------------------------------------------------
    print("\nFinal results (mean ± 95% CI over", n_runs, "runs):")
    print(df.to_string(index=False))
